# Deterministic Round-Trip Grid Search

This notebook runs round-trip tests for `DeterministicLLMCodec` across a grid of:
- `use_batch_invariant_ops`,
- `slots`,
- `logit_round_decimals`,
- `prob_round_decimals`.

For each parameter setting, it runs two routes:
- **Control:** CPU encode + CPU decode
- **Target:** CPU encode + GPU decode

For each test case, the model is freshly loaded and then explicitly released to control memory.

In [1]:
import gc
import time
import traceback
from itertools import product

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from llm_codec_deterministic import DeterministicLLMCodec, DeterministicCodecConfig

/homes/rm2092/LLM_Text_Compression_IIB_Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== User-configurable settings =====
MODEL_ID = "deepseek-ai/deepseek-coder-1.3b-base"
MAX_DECODE_TOKENS = 32  # fail fast if EOF is not reached
ENCODE_TIMEOUT_SEC = 120
DECODE_TIMEOUT_SEC = 60

# Keep this modest to avoid very long runs.
TEST_TEXTS = [
    "The quick brown fox jumps over the lazy dog."
]

# Grid toggles
USE_BATCH_INVARIANT_OPS_VALUES = [False, True]
SLOTS_VALUES = [32768, 65536, 131072]  # add more values to sweep count-resolution quantization
ROUND_DECIMALS_VALUES = [2, 3, 4]  # used for both logit and prob rounding

QUANT = True  # set False if you want no rounding at all

if torch.cuda.is_available():
    GPU_DEVICE = "cuda"
elif torch.backends.mps.is_available():
    GPU_DEVICE = "mps"
else:
    GPU_DEVICE = None

print("cuda available:", torch.cuda.is_available())
print("mps available:", torch.backends.mps.is_available())
print("decode device:", GPU_DEVICE)
print("encode timeout (sec):", ENCODE_TIMEOUT_SEC)
print("decode timeout (sec):", DECODE_TIMEOUT_SEC)
print("max decode tokens:", MAX_DECODE_TOKENS)
print("slots values:", SLOTS_VALUES)

if GPU_DEVICE is None:
    raise RuntimeError("No GPU backend detected (CUDA/MPS). This notebook requires GPU decode tests.")

cuda available: True
mps available: False
decode device: cuda
encode timeout (sec): 120
decode timeout (sec): 60
max decode tokens: 32
slots values: [32768, 65536, 131072]


In [3]:
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def _run_with_timeout(fn, timeout_sec, *args, **kwargs):
    if timeout_sec is None or timeout_sec <= 0:
        return fn(*args, **kwargs)

    # Hard wall-clock timeout (POSIX only).
    import os
    if os.name != "posix":
        return fn(*args, **kwargs)

    import signal

    class _CellTimeoutError(TimeoutError):
        pass

    def _handler(signum, frame):
        raise _CellTimeoutError(f"Timed out after {timeout_sec}s")

    old_handler = signal.getsignal(signal.SIGALRM)
    signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, float(timeout_sec))
    try:
        return fn(*args, **kwargs)
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, old_handler)


def run_one_case(text, use_batch_invariant_ops, slots, logit_round_decimals, prob_round_decimals, decode_device):
    started = time.time()

    tokenizer = None
    model = None
    encoder = None
    decoder = None
    encoded = None

    try:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)

        cfg = DeterministicCodecConfig(
            precision=32,
            slots=slots,
            use_kv_cache=False,
            quant=QUANT,
            logit_round_decimals=logit_round_decimals,
            prob_round_decimals=prob_round_decimals,
            use_batch_invariant_ops=use_batch_invariant_ops,
        )

        encoder = DeterministicLLMCodec(tokenizer, model, device="cpu", config=cfg)
        encoded = _run_with_timeout(encoder.encode, ENCODE_TIMEOUT_SEC, text)

        decoder = DeterministicLLMCodec(tokenizer, model, device=decode_device, config=cfg)
        decoded = _run_with_timeout(
            decoder.decode,
            DECODE_TIMEOUT_SEC,
            encoded,
            max_decode_tokens=MAX_DECODE_TOKENS,
        )

        original_bytes = len(text.encode("utf-8"))
        ratio_bits_per_byte = (len(encoded) * 8) / max(original_bytes, 1)

        return {
            "ok": decoded == text,
            "stage": "done",
            "error": "",
            "encoded_bytes": len(encoded),
            "original_bytes": original_bytes,
            "ratio_bits_per_byte": ratio_bits_per_byte,
            "elapsed_sec": round(time.time() - started, 3),
        }
    except Exception as exc:
        return {
            "ok": False,
            "stage": "encode" if encoded is None else "decode",
            "error": f"{type(exc).__name__}: {exc}",
            "encoded_bytes": None if encoded is None else len(encoded),
            "original_bytes": len(text.encode("utf-8")),
            "ratio_bits_per_byte": None if encoded is None else (len(encoded) * 8) / max(len(text.encode("utf-8")), 1),
            "elapsed_sec": round(time.time() - started, 3),
            "traceback_tail": "\n".join(traceback.format_exc().splitlines()[-8:]),
        }
    finally:
        del decoder
        del encoder
        del model
        del tokenizer
        clear_memory()

In [17]:
rows = []

grid = list(product(
    USE_BATCH_INVARIANT_OPS_VALUES,
    SLOTS_VALUES,
    ROUND_DECIMALS_VALUES,
))

decode_devices = ["cpu", GPU_DEVICE]
decode_devices = list(dict.fromkeys(decode_devices))

print(
    f"Running {len(grid)} parameter combinations x {len(TEST_TEXTS)} texts x {len(decode_devices)} decode routes "
    f"= {len(grid) * len(TEST_TEXTS) * len(decode_devices)} tests"
)

for use_batch_invariant_ops, slots, round_decimals in grid:
    for decode_device in decode_devices:
        for text_idx, text in enumerate(TEST_TEXTS):
            result = run_one_case(
                text=text,
                use_batch_invariant_ops=use_batch_invariant_ops,
                slots=slots,
                logit_round_decimals=round_decimals,
                prob_round_decimals=round_decimals,
                decode_device=decode_device,
            )

            route = f"cpu->{decode_device}"

            row = {
                "text_idx": text_idx,
                "text_preview": text[:60],
                "route": route,
                "decode_device": decode_device,
                "use_batch_invariant_ops": use_batch_invariant_ops,
                "slots": slots,
                "quant": QUANT,
                "logit_round_decimals": round_decimals,
                "prob_round_decimals": round_decimals,
                **result,
            }
            rows.append(row)

            print(
                f"text={text_idx} route={route} bi_ops={use_batch_invariant_ops} slots={slots} round={round_decimals} "
                f"ok={result['ok']} ratio={result['ratio_bits_per_byte']} time={result['elapsed_sec']}s"
            )

df = pd.DataFrame(rows)
df

Running 18 parameter combinations x 1 texts x 2 decode routes = 36 tests


Deterministic Encode: 100%|██████████| 13/13 [00:03<00:00,  3.86it/s]


text=0 route=cpu->cpu bi_ops=False slots=32768 round=2 ok=True ratio=3.4545454545454546 time=10.393s


Deterministic Encode: 100%|██████████| 13/13 [00:03<00:00,  3.89it/s]


text=0 route=cpu->cuda bi_ops=False slots=32768 round=2 ok=False ratio=3.4545454545454546 time=9.55s


...

text=0 route=cpu->cpu bi_ops=True slots=32768 round=3 ok=True ratio=3.8181818181818183 time=10.116s


Deterministic Encode: 100%|██████████| 13/13 [00:03<00:00,  3.82it/s]


text=0 route=cpu->cuda bi_ops=True slots=32768 round=3 ok=False ratio=3.8181818181818183 time=9.527s


Deterministic Encode: 100%|██████████| 13/13 [00:03<00:00,  3.81it/s]


KeyboardInterrupt: 

In [ ]:
# Summary views
ok_df = df[df["ok"] == True].copy()

print("Total tests:", len(df))
print("Passed:", int(df["ok"].sum()))
print("Failed:", int((~df["ok"]).sum()))

if len(ok_df) > 0:
    display(
        ok_df.groupby(["route", "use_batch_invariant_ops", "slots", "logit_round_decimals", "prob_round_decimals"], as_index=False)
            .agg(
                mean_ratio_bits_per_byte=("ratio_bits_per_byte", "mean"),
                max_ratio_bits_per_byte=("ratio_bits_per_byte", "max"),
                mean_elapsed_sec=("elapsed_sec", "mean"),
                n=("ok", "count"),
            )
            .sort_values(["mean_ratio_bits_per_byte", "mean_elapsed_sec"], ascending=[True, True])
    )

fail_df = df[df["ok"] == False].copy()
if len(fail_df) > 0:
    print("\nFailures (top 10):")
    display(fail_df[["text_idx", "route", "use_batch_invariant_ops", "slots", "logit_round_decimals", "prob_round_decimals", "error"]].head(10))

Total tests: 12
Passed: 6
Failed: 6


,route,use_batch_invariant_ops,logit_round_decimals,prob_round_decimals,mean_ratio_bits_per_byte,max_ratio_bits_per_byte,mean_elapsed_sec,n
5,cpu->cpu,True,4,4,1.272727,1.272727,9.942,1
2,cpu->cpu,False,4,4,1.272727,1.272727,10.179,1
1,cpu->cpu,False,3,3,1.454545,1.454545,10.156,1
4,cpu->cpu,True,3,3,1.454545,1.454545,10.246,1
0,cpu->cpu,False,2,2,1.818182,1.818182,10.123,1
3,cpu->cpu,True,2,2,1.818182,1.818182,10.128,1



Failures (top 10):


,text_idx,route,use_batch_invariant_ops,logit_round_decimals,prob_round_decimals,error
1,0,cpu->cuda,False,2,2,RuntimeError: Decoding exceeded max_decode_tok...
3,0,cpu->cuda,False,3,3,RuntimeError: Decoding exceeded max_decode_tok...
5,0,cpu->cuda,False,4,4,RuntimeError: Decoding exceeded max_decode_tok...
7,0,cpu->cuda,True,2,2,RuntimeError: Decoding exceeded max_decode_tok...
9,0,cpu->cuda,True,3,3,RuntimeError: Decoding exceeded max_decode_tok...
11,0,cpu->cuda,True,4,4,RuntimeError: Decoding exceeded max_decode_tok...


In [ ]:
# Quick sanity check: quant=True, batch-invariant ops disabled (CPU->CPU control)
quick_text = "The quick brown fox jumps over the lazy dog."

quick_result = run_one_case(
    text=quick_text,
    use_batch_invariant_ops=False,
    slots=SLOTS_VALUES[0],
    logit_round_decimals=2,
    prob_round_decimals=2,
    decode_device="cpu",
)

print("Quick sanity result:")
print(quick_result)

if quick_result.get("ok") and quick_result.get("ratio_bits_per_byte") is not None:
    print(f"ratio_bits_per_byte={quick_result['ratio_bits_per_byte']:.4f}")

Deterministic Encode: 100%|██████████| 13/13 [00:03<00:00,  3.93it/s]


Quick sanity result:
{'ok': True, 'stage': 'done', 'error': '', 'encoded_bytes': 10, 'original_bytes': 44, 'ratio_bits_per_byte': 1.8181818181818181, 'elapsed_sec': 10.124}
ratio_bits_per_byte=1.8182


In [ ]:
# CPU vs CUDA encode parity sweep (SHA256 over encoded bytes)
import hashlib

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available for this parity sweep.")

SHA_TEST_TEXT = "The quick brown fox jumps over the lazy dog."
PROB_ROUND_SWEEP = [None, 3, 2, 1, 0]  # None = quant disabled
SLOTS_SWEEP = [16384]  # add more values to sweep count-resolution quantization

tokenizer_cpu = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer_cuda = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_cuda = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)

rows_sha = []

for slots in SLOTS_SWEEP:
    for prob_rd in PROB_ROUND_SWEEP:
        quant = prob_rd is not None
        prob_d = 0 if prob_rd is None else prob_rd

        cfg = DeterministicCodecConfig(
            precision=32,
            slots=slots,
            use_kv_cache=False,
            quant=quant,
            logit_round_decimals=0,  # ignored in deterministic quant path
            prob_round_decimals=prob_d,
            use_batch_invariant_ops=False,
        )

        cpu_codec = DeterministicLLMCodec(tokenizer_cpu, model_cpu, device="cpu", config=cfg)
        cuda_codec = DeterministicLLMCodec(tokenizer_cuda, model_cuda, device="cuda", config=cfg)

        encoded_cpu = cpu_codec.encode(SHA_TEST_TEXT)
        encoded_cuda = cuda_codec.encode(SHA_TEST_TEXT)

        sha_cpu = hashlib.sha256(encoded_cpu).hexdigest()
        sha_cuda = hashlib.sha256(encoded_cuda).hexdigest()
        match = sha_cpu == sha_cuda

        # Optional direct cross-device decode check (CPU-encoded bytes decoded on CUDA)
        try:
            decoded_cuda = cuda_codec.decode(encoded_cpu, max_decode_tokens=64)
            cross_device_decode_ok = (decoded_cuda == SHA_TEST_TEXT)
            cross_device_error = ""
        except Exception as exc:
            cross_device_decode_ok = False
            cross_device_error = f"{type(exc).__name__}: {exc}"

        rows_sha.append({
            "slots": slots,
            "quant": quant,
            "prob_round_decimals": prob_rd,
            "encoded_len_cpu": len(encoded_cpu),
            "encoded_len_cuda": len(encoded_cuda),
            "sha_cpu": sha_cpu,
            "sha_cuda": sha_cuda,
            "sha_match": match,
            "cross_device_decode_ok": cross_device_decode_ok,
            "cross_device_error": cross_device_error,
        })

        del cuda_codec
        del cpu_codec
        clear_memory()

del model_cuda
del tokenizer_cuda
del model_cpu
del tokenizer_cpu
clear_memory()

df_sha = pd.DataFrame(rows_sha)
display(df_sha[[
    "slots", "quant", "prob_round_decimals", "encoded_len_cpu", "encoded_len_cuda",
    "sha_match", "cross_device_decode_ok", "cross_device_error"
]])

Loading weights: 100%|██████████| 219/219 [00:00<00:00, 798.93it/s, Materializing param=model.norm.weight]                              


Deterministic Encode: 100%|██████████| 13/13 [00:00<00:00, 15.93it/s]


,slots,quant,prob_round_decimals,encoded_len_cpu,encoded_len_cuda,sha_match,cross_device_decode_ok,cross_device_error
0,32768,False,NaN,21,21,False,False,RuntimeError: Decoding exceeded max_decode_tok...
1,32768,True,3.0,22,21,False,False,RuntimeError: Decoding exceeded max_decode_tok...
2,32768,True,2.0,20,20,False,False,RuntimeError: Decoding exceeded max_decode_tok...
3,32768,True,1.0,15,15,True,True,
4,32768,True,0.0,16,16,True,True,
5,65536,False,NaN,11,11,False,False,RuntimeError: Decoding exceeded max_decode_tok...
6,65536,True,3.0,11,11,False,False,RuntimeError: Decoding exceeded max_decode_tok...
7,65536,True,2.0,12,12,False,False,RuntimeError: Decoding exceeded max_decode_tok...
8,65536,True,1.0,10,9,False,False,RuntimeError: Decoding exceeded max_decode_tok...
9,65536,True,0.0,13,13,True,True,


In [4]:
# CPU vs CUDA parity sweep at fixed slots=32768, with/without batch_invariant_ops
import hashlib

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available for this parity sweep.")

SHA_TEST_TEXT = "The quick brown fox jumps over the lazy dog."
FIXED_SLOTS = 32768
PROB_ROUND_SWEEP = [None, 3, 2, 1, 0]  # None = quant disabled
BI_OPS_SWEEP = [False, True]

tokenizer_cpu = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_cpu = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer_cuda = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_cuda = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True)

rows_sha_bi = []

for use_bi_ops in BI_OPS_SWEEP:
    for prob_rd in PROB_ROUND_SWEEP:
        quant = prob_rd is not None
        prob_d = 0 if prob_rd is None else prob_rd

        cfg = DeterministicCodecConfig(
            precision=32,
            slots=FIXED_SLOTS,
            use_kv_cache=False,
            quant=quant,
            logit_round_decimals=0,  # ignored in deterministic quant path
            prob_round_decimals=prob_d,
            use_batch_invariant_ops=use_bi_ops,
        )

        cpu_codec = DeterministicLLMCodec(tokenizer_cpu, model_cpu, device="cpu", config=cfg)
        cuda_codec = DeterministicLLMCodec(tokenizer_cuda, model_cuda, device="cuda", config=cfg)

        encoded_cpu = cpu_codec.encode(SHA_TEST_TEXT)
        encoded_cuda = cuda_codec.encode(SHA_TEST_TEXT)

        sha_cpu = hashlib.sha256(encoded_cpu).hexdigest()
        sha_cuda = hashlib.sha256(encoded_cuda).hexdigest()
        match = sha_cpu == sha_cuda

        try:
            decoded_cuda = cuda_codec.decode(encoded_cpu, max_decode_tokens=64)
            cross_device_decode_ok = (decoded_cuda == SHA_TEST_TEXT)
            cross_device_error = ""
        except Exception as exc:
            cross_device_decode_ok = False
            cross_device_error = f"{type(exc).__name__}: {exc}"

        rows_sha_bi.append({
            "slots": FIXED_SLOTS,
            "use_batch_invariant_ops": use_bi_ops,
            "quant": quant,
            "prob_round_decimals": prob_rd,
            "encoded_len_cpu": len(encoded_cpu),
            "encoded_len_cuda": len(encoded_cuda),
            "sha_cpu": sha_cpu,
            "sha_cuda": sha_cuda,
            "sha_match": match,
            "cross_device_decode_ok": cross_device_decode_ok,
            "cross_device_error": cross_device_error,
        })

        del cuda_codec
        del cpu_codec
        clear_memory()

del model_cuda
del tokenizer_cuda
del model_cpu
del tokenizer_cpu
clear_memory()

df_sha_bi = pd.DataFrame(rows_sha_bi)
display(df_sha_bi[[
    "slots", "use_batch_invariant_ops", "quant", "prob_round_decimals",
    "encoded_len_cpu", "encoded_len_cuda", "sha_match",
    "cross_device_decode_ok", "cross_device_error"
]])

Deterministic Encode: 100%|██████████| 13/13 [00:00<00:00, 16.50it/s]
/homes/rm2092/LLM_Text_Compression_IIB_Project/.venv/lib/python3.12/site-packages/torch/library.py:357: UserWarning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::mm(Tensor self, Tensor mat2) -> Tensor
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: CUDA
  previous kernel: registered at /pytorch/aten/src/ATen/LegacyBatchingRegistrations.cpp:1076
       new kernel: registered at /homes/rm2092/LLM_Text_Compression_IIB_Project/batch_invariant_ops/batch_invariant_ops/batch_invariant_ops.py:506 (Triggered internally at /pytorch/aten/src/ATen/core/dispatch/OperatorEntry.cpp:208.)
  self.m.impl(
Deterministic Encode: 100%|██████████| 13/13 [00:00<00:00, 16.62it/s]


,slots,use_batch_invariant_ops,quant,prob_round_decimals,encoded_len_cpu,encoded_len_cuda,sha_match,cross_device_decode_ok,cross_device_error
0,32768,False,False,NaN,21,21,False,False,RuntimeError: Decoding exceeded max_decode_tok...
1,32768,False,True,3.0,21,21,False,False,RuntimeError: Decoding exceeded max_decode_tok...
2,32768,False,True,2.0,19,18,False,False,RuntimeError: Decoding exceeded max_decode_tok...
3,32768,False,True,1.0,17,17,True,True,
4,32768,False,True,0.0,13,13,True,True,
5,32768,True,False,NaN,21,21,False,False,RuntimeError: Decoding exceeded max_decode_tok...
6,32768,True,True,3.0,21,21,False,False,RuntimeError: Decoding exceeded max_decode_tok...
7,32768,True,True,2.0,19,18,False,False,RuntimeError: Decoding exceeded max_decode_tok...
8,32768,True,True,1.0,17,17,True,True,
9,32768,True,True,0.0,13,13,True,True,


In [6]:
# Export portable .bin files + author-style batch-invariance check (same scale)
import json
import hashlib
import time
import traceback
from pathlib import Path

try:
    from batch_invariant_ops import set_batch_invariant_mode
except Exception:
    try:
        from batch_invariant_ops.batch_invariant_ops import set_batch_invariant_mode
    except Exception:
        set_batch_invariant_mode = None


def _author_style_test(mode_enabled: bool, iters: int = 3):
    if set_batch_invariant_mode is None:
        return {"mode": mode_enabled, "ran": False, "reason": "batch_invariant_ops import unavailable in notebook kernel"}

    # Same scale as author's PoC.
    B, D = 2048, 4096
    dtype = torch.float32
    diffs = []

    with set_batch_invariant_mode(mode_enabled):
        for _ in range(iters):
            a = torch.linspace(-100, 100, B * D, dtype=dtype).reshape(B, D)
            b = torch.linspace(-100, 100, D * D, dtype=dtype).reshape(D, D)
            out1 = torch.mm(a[:1], b)
            out2 = torch.mm(a, b)[:1]
            diff = (out1 - out2).abs().max().item()
            diffs.append(diff)

    return {
        "mode": mode_enabled,
        "ran": True,
        "iters": iters,
        "max_diff": max(diffs),
        "min_diff": min(diffs),
        "all_zero": all(d == 0.0 for d in diffs),
    }


print("Author-style same-scale test:")
display(pd.DataFrame([_author_style_test(False), _author_style_test(True)]))

EXPORT_DIR = Path("project_files/cross_device_bins")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_MODEL_ID = MODEL_ID
ENCODE_DEVICE = "cpu"
ENCODE_TIMEOUT_SEC_EXPORT = 120

# You can tweak this list to generate more/fewer portable samples.
EXPORT_CASES = [
    {"label": "q0_bi0", "use_batch_invariant_ops": False, "quant": True, "prob_round_decimals": 0, "slots": 32768},
    {"label": "q1_bi0", "use_batch_invariant_ops": False, "quant": True, "prob_round_decimals": 1, "slots": 32768},
    {"label": "q0_bi1", "use_batch_invariant_ops": True,  "quant": True, "prob_round_decimals": 0, "slots": 32768},
    {"label": "q1_bi1", "use_batch_invariant_ops": True,  "quant": True, "prob_round_decimals": 1, "slots": 32768},
]

TEXT_SAMPLES = [
    "The quick brown fox jumps over the lazy dog.",
    "A short deterministic codec portability test.",
]


def _run_with_timeout_local(fn, timeout_sec, *args, **kwargs):
    if timeout_sec is None or timeout_sec <= 0:
        return fn(*args, **kwargs)
    import os
    if os.name != "posix":
        return fn(*args, **kwargs)
    import signal

    class _TimeoutErrorLocal(TimeoutError):
        pass

    def _handler(signum, frame):
        raise _TimeoutErrorLocal(f"Timed out after {timeout_sec}s")

    old_handler = signal.getsignal(signal.SIGALRM)
    signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, float(timeout_sec))
    try:
        return fn(*args, **kwargs)
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, old_handler)


manifest = {
    "model_id": EXPORT_MODEL_ID,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "encode_device": ENCODE_DEVICE,
    "cases": [],
}

rows_export = []
for case in EXPORT_CASES:
    for text_idx, text in enumerate(TEXT_SAMPLES):
        case_id = f"{case['label']}_t{text_idx}"
        bin_name = f"{case_id}.bin"
        bin_path = EXPORT_DIR / bin_name

        tokenizer = None
        model = None
        codec = None
        encoded = None
        started = time.time()

        try:
            tokenizer = AutoTokenizer.from_pretrained(EXPORT_MODEL_ID, trust_remote_code=True)
            model = AutoModelForCausalLM.from_pretrained(EXPORT_MODEL_ID, trust_remote_code=True)

            cfg = DeterministicCodecConfig(
                precision=32,
                slots=int(case["slots"]),
                use_kv_cache=False,
                quant=bool(case["quant"]),
                logit_round_decimals=0,
                prob_round_decimals=int(case["prob_round_decimals"]),
                use_batch_invariant_ops=bool(case["use_batch_invariant_ops"]),
            )

            codec = DeterministicLLMCodec(tokenizer, model, device=ENCODE_DEVICE, config=cfg)
            encoded = _run_with_timeout_local(codec.encode, ENCODE_TIMEOUT_SEC_EXPORT, text)

            with open(bin_path, "wb") as f:
                f.write(encoded)

            record = {
                "case_id": case_id,
                "bin_file": bin_name,
                "text_idx": text_idx,
                "expected_text": text,
                "expected_text_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
                "encoded_sha256": hashlib.sha256(encoded).hexdigest(),
                "encoded_len": len(encoded),
                "slots": int(case["slots"]),
                "quant": bool(case["quant"]),
                "prob_round_decimals": int(case["prob_round_decimals"]),
                "use_batch_invariant_ops": bool(case["use_batch_invariant_ops"]),
                "elapsed_sec": round(time.time() - started, 3),
            }
            manifest["cases"].append(record)
            rows_export.append({**record, "ok": True, "error": ""})
        except Exception as exc:
            rows_export.append({
                "case_id": case_id,
                "bin_file": bin_name,
                "text_idx": text_idx,
                "slots": int(case["slots"]),
                "quant": bool(case["quant"]),
                "prob_round_decimals": int(case["prob_round_decimals"]),
                "use_batch_invariant_ops": bool(case["use_batch_invariant_ops"]),
                "ok": False,
                "error": f"{type(exc).__name__}: {exc}",
                "elapsed_sec": round(time.time() - started, 3),
                "traceback_tail": "\n".join(traceback.format_exc().splitlines()[-8:]),
            })
        finally:
            del codec
            del model
            del tokenizer
            clear_memory()

manifest_path = EXPORT_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Saved bins + manifest to: {EXPORT_DIR.resolve()}")
df_export = pd.DataFrame(rows_export)
display(df_export[[
    "case_id", "bin_file", "slots", "quant", "prob_round_decimals",
    "use_batch_invariant_ops", "ok", "encoded_len", "elapsed_sec", "error"
]])

Author-style same-scale test:


,mode,ran,iters,max_diff,min_diff,all_zero
0,False,True,3,8.75,8.75,False
1,True,True,3,8.75,8.75,False


Deterministic Encode: 100%|██████████| 11/11 [00:02<00:00,  4.39it/s]


Saved bins + manifest to: /homes/rm2092/LLM_Text_Compression_IIB_Project/project_files/cross_device_bins


,case_id,bin_file,slots,quant,prob_round_decimals,use_batch_invariant_ops,ok,encoded_len,elapsed_sec,error
0,q0_bi0_t0,q0_bi0_t0.bin,32768,True,0,False,True,13,6.998,
1,q0_bi0_t1,q0_bi0_t1.bin,32768,True,0,False,True,19,6.147,
2,q1_bi0_t0,q1_bi0_t0.bin,32768,True,1,False,True,17,6.835,
3,q1_bi0_t1,q1_bi0_t1.bin,32768,True,1,False,True,21,6.076,
4,q0_bi1_t0,q0_bi1_t0.bin,32768,True,0,True,True,13,6.874,
5,q0_bi1_t1,q0_bi1_t1.bin,32768,True,0,True,True,19,5.692,
6,q1_bi1_t0,q1_bi1_t0.bin,32768,True,1,True,True,17,6.652,
7,q1_bi1_t1,q1_bi1_t1.bin,32768,True,1,True,True,21,5.927,


In [7]:
# Decode portable .bin files with safety limits (run this on another device if desired)
import json
import hashlib
import time
import traceback
from pathlib import Path

DECODE_DIR = Path("project_files/cross_device_bins")
MANIFEST_PATH = DECODE_DIR / "manifest.json"
DECODE_TIMEOUT_SEC_PORTABLE = 120
MAX_DECODE_TOKENS_PORTABLE = 128

if torch.cuda.is_available():
    DECODE_DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DECODE_DEVICE = "mps"
else:
    DECODE_DEVICE = "cpu"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

decode_model_id = manifest.get("model_id", MODEL_ID)
records = manifest.get("cases", [])
print(f"Decode device: {DECODE_DEVICE}")
print(f"Model: {decode_model_id}")
print(f"Cases: {len(records)}")


def _run_with_timeout_local(fn, timeout_sec, *args, **kwargs):
    if timeout_sec is None or timeout_sec <= 0:
        return fn(*args, **kwargs)
    import os
    if os.name != "posix":
        return fn(*args, **kwargs)
    import signal

    class _TimeoutErrorLocal(TimeoutError):
        pass

    def _handler(signum, frame):
        raise _TimeoutErrorLocal(f"Timed out after {timeout_sec}s")

    old_handler = signal.getsignal(signal.SIGALRM)
    signal.signal(signal.SIGALRM, _handler)
    signal.setitimer(signal.ITIMER_REAL, float(timeout_sec))
    try:
        return fn(*args, **kwargs)
    finally:
        signal.setitimer(signal.ITIMER_REAL, 0.0)
        signal.signal(signal.SIGALRM, old_handler)


def _first_mismatch(a: str, b: str):
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] != b[i]:
            return i
    if len(a) != len(b):
        return n
    return -1


def _snippet(s: str, idx: int, radius: int = 24):
    if idx < 0:
        return ""
    start = max(0, idx - radius)
    end = min(len(s), idx + radius)
    return s[start:end]


rows_decode = []
for item in records:
    started = time.time()
    tokenizer = None
    model = None
    codec = None

    case_id = item.get("case_id", "unknown")
    bin_file = item.get("bin_file")
    expected_text = item.get("expected_text", "")
    bin_path = DECODE_DIR / str(bin_file)

    try:
        if not bin_path.exists():
            raise FileNotFoundError(f"Missing bin file: {bin_path}")

        with open(bin_path, "rb") as f:
            encoded_bytes = f.read()

        cfg = DeterministicCodecConfig(
            precision=32,
            slots=int(item["slots"]),
            use_kv_cache=False,
            quant=bool(item["quant"]),
            logit_round_decimals=0,
            prob_round_decimals=int(item["prob_round_decimals"]),
            use_batch_invariant_ops=bool(item["use_batch_invariant_ops"]),
        )

        tokenizer = AutoTokenizer.from_pretrained(decode_model_id, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(decode_model_id, trust_remote_code=True)
        codec = DeterministicLLMCodec(tokenizer, model, device=DECODE_DEVICE, config=cfg)

        decoded_text = _run_with_timeout_local(
            codec.decode,
            DECODE_TIMEOUT_SEC_PORTABLE,
            encoded_bytes,
            max_decode_tokens=MAX_DECODE_TOKENS_PORTABLE,
        )

        expected_sha = hashlib.sha256(expected_text.encode("utf-8")).hexdigest()
        decoded_sha = hashlib.sha256(decoded_text.encode("utf-8")).hexdigest()
        mismatch_pos = _first_mismatch(expected_text, decoded_text)

        rows_decode.append({
            "case_id": case_id,
            "bin_file": bin_file,
            "decode_device": DECODE_DEVICE,
            "slots": int(item["slots"]),
            "quant": bool(item["quant"]),
            "prob_round_decimals": int(item["prob_round_decimals"]),
            "use_batch_invariant_ops": bool(item["use_batch_invariant_ops"]),
            "encoded_len": len(encoded_bytes),
            "expected_len": len(expected_text),
            "decoded_len": len(decoded_text),
            "first_mismatch_pos": mismatch_pos,
            "expected_snippet": _snippet(expected_text, mismatch_pos),
            "decoded_snippet": _snippet(decoded_text, mismatch_pos),
            "expected_sha256": expected_sha,
            "decoded_sha256": decoded_sha,
            "text_match": decoded_text == expected_text,
            "elapsed_sec": round(time.time() - started, 3),
            "error": "",
        })
    except Exception as exc:
        rows_decode.append({
            "case_id": case_id,
            "bin_file": bin_file,
            "decode_device": DECODE_DEVICE,
            "slots": int(item.get("slots", -1)),
            "quant": bool(item.get("quant", False)),
            "prob_round_decimals": int(item.get("prob_round_decimals", -1)),
            "use_batch_invariant_ops": bool(item.get("use_batch_invariant_ops", False)),
            "encoded_len": None if not bin_path.exists() else len(open(bin_path, "rb").read()),
            "expected_len": len(expected_text),
            "decoded_len": None,
            "first_mismatch_pos": None,
            "expected_snippet": "",
            "decoded_snippet": "",
            "expected_sha256": hashlib.sha256(expected_text.encode("utf-8")).hexdigest(),
            "decoded_sha256": "",
            "text_match": False,
            "elapsed_sec": round(time.time() - started, 3),
            "error": f"{type(exc).__name__}: {exc}",
            "traceback_tail": "\n".join(traceback.format_exc().splitlines()[-8:]),
        })
    finally:
        del codec
        del model
        del tokenizer
        clear_memory()

df_decode_portable = pd.DataFrame(rows_decode)
display(df_decode_portable[[
    "case_id", "bin_file", "decode_device", "slots", "quant",
    "prob_round_decimals", "use_batch_invariant_ops",
    "encoded_len", "expected_len", "decoded_len", "first_mismatch_pos",
    "text_match", "elapsed_sec", "error"
]])

mismatch_df = df_decode_portable[df_decode_portable["text_match"] == False].copy()
if len(mismatch_df) > 0:
    print("\nMismatch details (up to 10):")
    display(mismatch_df[[
        "case_id", "first_mismatch_pos", "expected_snippet", "decoded_snippet", "error"
    ]].head(10))

print("\nPass count:", int(df_decode_portable["text_match"].sum()), "/", len(df_decode_portable))

Decode device: cuda
Model: deepseek-ai/deepseek-coder-1.3b-base
Cases: 8


Loading weights: 100%|██████████| 219/219 [00:00<00:00, 679.37it/s, Materializing param=model.norm.weight]                              


,case_id,bin_file,decode_device,slots,quant,prob_round_decimals,use_batch_invariant_ops,encoded_len,expected_len,decoded_len,first_mismatch_pos,text_match,elapsed_sec,error
0,q0_bi0_t0,q0_bi0_t0.bin,cuda,32768,True,0,False,13,44,44,-1,True,3.022,
1,q0_bi0_t1,q0_bi0_t1.bin,cuda,32768,True,0,False,19,45,45,-1,True,2.817,
2,q1_bi0_t0,q1_bi0_t0.bin,cuda,32768,True,1,False,17,44,44,-1,True,2.892,
3,q1_bi0_t1,q1_bi0_t1.bin,cuda,32768,True,1,False,21,45,45,-1,True,2.843,
4,q0_bi1_t0,q0_bi1_t0.bin,cuda,32768,True,0,True,13,44,44,-1,True,2.936,
5,q0_bi1_t1,q0_bi1_t1.bin,cuda,32768,True,0,True,19,45,45,-1,True,2.746,
6,q1_bi1_t0,q1_bi1_t0.bin,cuda,32768,True,1,True,17,44,44,-1,True,2.830,
7,q1_bi1_t1,q1_bi1_t1.bin,cuda,32768,True,1,True,21,45,45,-1,True,2.697,



Pass count: 8 / 8
